# 🗂️ Notebook 2: Netflix — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Core entities

| Table | Key columns | Storage |
|---|---|---|
| `users` | id, email, region | relational (MySQL/Postgres) |
| `titles` | id, name, synopsis, genres, release_year | relational |
| `video_assets` | id, title_id, language | relational |
| `renditions` | video_asset_id, resolution, bitrate_kbps, cdn_url | relational |
| `watch_history` | user_id, title_id, position_seconds, updated_at | **Cassandra** (high write) |
| `subscriptions` | user_id, tier, valid_until | relational |

### Why split `titles` from `video_assets` from `renditions`?
- A **title** is the creative work ("The Matrix").
- An **asset** is a specific cut+language ("The Matrix, Japanese dub, director's cut").
- A **rendition** is one encoding of an asset ("720p, 2.8 Mbps, HLS chunks on CDN").

Keeping them separate means adding a new language doesn't duplicate metadata, and adding
a new bitrate doesn't touch the title row.


## Bad → Best: designing `watch_history`

This table is surprisingly tricky. Let's iterate on it.

In [ ]:
# --- v1 (BAD): append every position update as a new row ---
# INSERT INTO watch_history (user_id, title_id, position_sec, ts) VALUES (...)
# Problem: if the client checkpoints every 5 seconds during a 2-hour movie,
# that is 1,440 rows per view. Multiply by 30M concurrent viewers = a write storm.
writes_per_view = (2 * 60 * 60) // 5
peak_views = 30_000_000
print(f"v1 writes/sec at peak: {writes_per_view * peak_views / (2*60*60):,.0f}")  # ~6M writes/sec

# --- v2 (BETTER): upsert on (user_id, title_id) - only latest position matters ---
# UPSERT watch_history SET position_sec=?, ts=? WHERE user_id=? AND title_id=?
# One row per (user, title). Resume-where-you-left-off only needs the latest.

# --- v3 (BEST): batch + async, eventually consistent ---
# Client sends checkpoints every 30s -> gateway batches -> Kafka -> Cassandra upsert.
# - Reads (resume) go through a small Redis cache; miss falls back to Cassandra.
# - Writes are idempotent by (user_id, title_id) and tolerate retries.
# - Cross-region replication: last-write-wins on ts is fine for a position counter.
print("v3: writes are bounded by CLIENT CHECKPOINT INTERVAL, not per-row appends.")


## Key APIs

```http
# Browse
GET  /catalog/home              -> personalized rails
GET  /titles/{id}               -> title details
GET  /search?q=...

# Playback handshake
POST /playback/start            { title_id }
     -> { manifest_url, session_token, initial_position }

# The *player* then fetches manifest_url from the CDN,
# and the CDN streams HLS chunks directly.

# Position checkpoints (low-priority, batched)
POST /history/position          { title_id, position_seconds }

# Recommendations
GET  /recs/for-me               -> ranked list of titles
```

**Important**: the server returns a **manifest URL** pointing to a CDN, not raw bytes.
Our origin servers never stream video.


In [ ]:
# A minimal pydantic model of the API surface - runs without a server.
from pydantic import BaseModel, Field
from datetime import datetime

class Rendition(BaseModel):
    resolution: str
    bitrate_kbps: int
    cdn_url: str

class Title(BaseModel):
    id: int
    name: str
    genres: list[str]
    release_year: int

class PlaybackStartRequest(BaseModel):
    title_id: int

class PlaybackSession(BaseModel):
    session_token: str
    manifest_url: str
    initial_position_sec: int = 0

class PositionCheckpoint(BaseModel):
    title_id: int
    position_seconds: int = Field(ge=0)
    ts: datetime

sess = PlaybackSession(
    session_token="sess-abc",
    manifest_url="https://cdn.example.com/m/42.m3u8",
    initial_position_sec=120,
)
print(sess.model_dump_json(indent=2))


## An HLS manifest (what the CDN actually returns)

The *master* playlist is a tiny text file. The player downloads it first, then picks a
variant based on its measured bandwidth.

In [ ]:
# A tiny HLS master manifest parser - no external libs.
master = """#EXTM3U
#EXT-X-VERSION:3
#EXT-X-STREAM-INF:BANDWIDTH=800000,RESOLUTION=640x360
360p.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=2800000,RESOLUTION=1280x720
720p.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=5000000,RESOLUTION=1920x1080
1080p.m3u8
"""

def parse_master(text: str):
    variants = []
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    i = 0
    while i < len(lines):
        if lines[i].startswith("#EXT-X-STREAM-INF"):
            attrs = dict(kv.split("=", 1) for kv in lines[i].split(":", 1)[1].split(","))
            variants.append({
                "bandwidth": int(attrs["BANDWIDTH"]),
                "resolution": attrs["RESOLUTION"],
                "uri": lines[i + 1],
            })
            i += 2
        else:
            i += 1
    return variants

for v in parse_master(master):
    print(v)


In [ ]:
# Which variant would a player pick given a measured bandwidth?
def pick_variant(variants, measured_bps, safety=0.8):
    """Pick the highest-bandwidth variant that fits within safety*measured_bps."""
    budget = measured_bps * safety
    feasible = [v for v in variants if v["bandwidth"] <= budget]
    return max(feasible, key=lambda v: v["bandwidth"]) if feasible else min(variants, key=lambda v: v["bandwidth"])

variants = parse_master(master)
for mbps in (0.5, 2, 4, 10):
    v = pick_variant(variants, mbps * 1_000_000)
    print(f"{mbps:>5} Mbps network -> {v['resolution']} @ {v['bandwidth']/1e6:.1f} Mbps")


## Takeaways
1. **Normalize writes that grow unbounded.** `watch_history` as upsert-by-(user,title) beats
   append-only by ~1000x.
2. **APIs return manifest URLs, not bytes.** Origin never streams video.
3. **ABR is a client-side decision** driven by a tiny text playlist. The server's job is to
   publish the right variants.
